In [1]:
# 13_realistic_nonmemorized_carrier.ipynb -- Tighten the non-memorized reconstruction bracket [0.006, 0.18].
# Reviewer concern (Round-1 Major 5): the whole "novel code is highly protected" conclusion rests on ONE
# synthetic estimate (strong-adversary 0.006) from a carrier the paper admits is simpler than real code and
# therefore may UNDER-state reconstruction. The paper itself flags "a non-memorized carrier closer to real
# proprietary code (richer inter-identifier context)" as the key remaining measurement (Appendix A).
#
# This notebook runs that measurement with TWO carrier sources, scored under ONE matched near-match rule
# against the cached PUBLIC Fineract baseline, so the three points are directly comparable:
#   MODE="private"   -> point PRIVATE_DIR at a genuinely novel, non-sensitive codebase of your own (BEST:
#                       real inter-identifier context, guaranteed non-memorized if it was never published).
#   MODE="synthetic" -> a RICHER synthetic generator than nb11: multi-method classes with cross-calls,
#                       field access, and dependency between identifiers, so context-based reconstruction
#                       is a fairer (harder) test than the original logistics generator.
# For each carrier it: (verbatim probe) -> C2 identifier mask -> adversary (exact + near) with bootstrap CIs
# -> re-score the public baseline from the SAME near-match rule -> report the memorization share.
#
# HOW TO RUN
#   1) Notebooks 01-02 must have been run once (data/*.json exist) for the public baseline.
#   2) Copy .env.example -> .env; set ONE vendor key and the strong model id.
#   3a) Private mode: set PRIVATE_DIR below to a folder of .java (or .py) files that were NEVER published
#       and contain NO real secrets/PII (this notebook injects synthetic tokens itself).
#   3b) Synthetic mode: leave PRIVATE_DIR empty; a richer synthetic carrier is generated here.
#   4) Run. Outputs: results/tables/nonmemorized_carrier.csv. Paths are not printed (anonymity).
#
# No proprietary data leaves your machine except the masked carrier sent to the adversary model you chose,
# exactly as in the paper's threat model. If that is a concern, use the synthetic mode.

import os, re, json, time, random, hashlib, difflib, urllib.request, urllib.error
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED); rng = np.random.default_rng(SEED)

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA, TAB, CACHE = ROOT / "data", ROOT / "results" / "tables", ROOT / "data" / "_adv_cache"
for d in (TAB, CACHE):
    d.mkdir(parents=True, exist_ok=True)

# ------------------------- CONFIG -------------------------
PRIVATE_DIR = ""                   # <-- set to a path for private mode; leave "" for synthetic mode
LANG        = "java"               # "java" or "python" (must match PRIVATE_DIR files if private mode)
N_FILES     = 40                   # carrier files to evaluate
K_IDENTS    = 12
NEAR        = 0.8
BOOT        = 10000
# ----------------------------------------------------------

def load_env(p):
    cfg = {}
    if p.exists():
        for ln in p.read_text(encoding="utf-8").splitlines():
            ln = ln.strip()
            if ln and not ln.startswith("#") and "=" in ln:
                k, v = ln.split("=", 1); cfg[k.strip()] = v.strip().strip('"').strip("'")
    return cfg
ENV = load_env(ROOT / ".env")
def env(k, d=""): return ENV.get(k, os.environ.get(k, d))

PRIVATE_DIR = env("PRIVATE_DIR", PRIVATE_DIR)
LANG        = env("SYN_LANG", LANG)
N_FILES     = int(env("SYN_N", str(N_FILES)))

# Single strong adversary (the bracket is reported for the strong tier). Pick the vendor you have.
PROVIDER = env("ADV_PROVIDER", "openai")        # openai | anthropic | google
MODEL = {
    "openai":    env("LLM_MODEL_STRONG", "gpt-5.4"),
    "anthropic": env("ANTHROPIC_MODEL_STRONG", "claude-sonnet-4-5"),
    "google":    env("GOOGLE_MODEL_STRONG", "gemini-2.5-pro"),
}[PROVIDER]
KEY = {
    "openai":    env("OPENAI_API_KEY"),
    "anthropic": env("ANTHROPIC_API_KEY"),
    "google":    env("GOOGLE_API_KEY") or env("GEMINI_API_KEY"),
}[PROVIDER]
USE_LLM = bool(KEY) and KEY.lower() != "none"

# ------------------ multi-provider chat + cache ------------------
def _http_json(url, headers, body, timeout=120):
    req = urllib.request.Request(url, data=json.dumps(body).encode(), headers=headers)
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())
def _openai(model, prompt, temp):
    body = {"model": model, "messages": [{"role": "user", "content": prompt}], "temperature": temp}
    hdr = {"Authorization": f"Bearer {KEY}", "Content-Type": "application/json"}
    try:
        d = _http_json("https://api.openai.com/v1/chat/completions", hdr, body)
    except urllib.error.HTTPError as e:
        if e.code == 400 and "temperature" in e.read().decode(errors="ignore"):
            body.pop("temperature"); d = _http_json("https://api.openai.com/v1/chat/completions", hdr, body)
        else: raise
    return d["choices"][0]["message"]["content"]
def _anthropic(model, prompt, temp):
    body = {"model": model, "max_tokens": 1024, "temperature": temp,
            "messages": [{"role": "user", "content": prompt}]}
    hdr = {"x-api-key": KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"}
    d = _http_json("https://api.anthropic.com/v1/messages", hdr, body)
    return "".join(b.get("text", "") for b in d.get("content", []) if b.get("type") == "text")
def _google(model, prompt, temp):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={KEY}"
    body = {"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"temperature": temp}}
    d = _http_json(url, {"Content-Type": "application/json"}, body)
    c = d.get("candidates", [])
    return "".join(p.get("text", "") for p in c[0]["content"]["parts"]) if c else ""
_DISP = {"openai": _openai, "anthropic": _anthropic, "google": _google}
def chat(prompt, temp=0.0, tag=""):
    key = hashlib.sha256(f"{PROVIDER}|{MODEL}|{temp}|{tag}|{prompt}".encode()).hexdigest()
    cf = CACHE / f"nm_{key}.json"
    if cf.exists():
        return json.loads(cf.read_text(encoding="utf-8"))["content"]
    for a in range(4):
        try:
            c = _DISP[PROVIDER](MODEL, prompt, temp)
            cf.write_text(json.dumps({"content": c}), encoding="utf-8"); return c
        except urllib.error.HTTPError as e:
            if e.code in (429, 500, 502, 503, 529) and a < 3: time.sleep(2 ** a); continue
            raise
    return ""

def parse_map(t):
    i, j = t.find("{"), t.rfind("}")
    if i == -1 or j == -1: return {}
    try: return json.loads(t[i:j + 1])
    except Exception: return {}
def norm(s): return str(s).strip().strip("<>").lower()
def near(a, b): return difflib.SequenceMatcher(None, norm(a), norm(b)).ratio() >= NEAR
def boot_ci(v):
    a = np.asarray(v, float); a = a[~np.isnan(a)]
    if len(a) == 0: return (float("nan"), float("nan"))
    m = a[rng.integers(0, len(a), size=(BOOT, len(a))).astype(int)].mean(axis=1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

# =========================================================================
# RICHER synthetic generator: classes with multiple methods that CALL EACH OTHER and share fields,
# so an identifier's meaning is constrained by cross-references -> harder, fairer reconstruction than nb11.
# =========================================================================
VERB = ["compute","validate","schedule","reconcile","dispatch","allocate","register","resolve","aggregate",
        "normalize","estimate","prioritize","archive","rebalance","assign","merge","settle","accrue","post","audit"]
NOUN = ["shipment","inventory","route","warehouse","order","pallet","courier","manifest","batch","slot","ledger",
        "carrier","parcel","depot","lane","invoice","dock","crate","supplier","consignment","tariff","account"]
ATTR = ["weight","volume","priority","status","capacity","distance","duration","count","ratio","threshold",
        "label","code","offset","margin","index","balance","rate","tier","cursor","bucket"]
def cap(s): return s[:1].upper() + s[1:]
def cc(*w): return w[0] + "".join(cap(x) for x in w[1:])
def snake(*w): return "_".join(w)

def gen_java(i):
    noun = random.choice(NOUN); cls = cap(noun) + random.choice(["Service","Manager","Engine","Coordinator"])
    fields = [cc(noun, random.choice(ATTR)) for _ in range(random.randint(3, 5))]
    L = [f"// domain service for {noun} handling", f"public class {cls} {{"]
    for f in fields:
        L.append(f"    private int {f};")
    helpers = []
    for _ in range(random.randint(2, 3)):               # helper methods that later methods call
        hv, hn = random.choice(VERB), random.choice(NOUN); hm = cc(hv, hn)
        p = cc(hn, random.choice(ATTR)); loc = cc(random.choice(VERB), random.choice(ATTR))
        helpers.append((hm, p))
        L += [f"    private int {hm}(int {p}) {{",
              f"        int {loc} = {p} + {random.choice(fields)};",
              f"        return {loc} % {random.randint(3, 9)};",
              f"    }}"]
    for _ in range(random.randint(3, 5)):               # public methods that call helpers and touch fields
        v, n2 = random.choice(VERB), random.choice(NOUN); meth = cc(v, n2)
        p = cc(n2, random.choice(ATTR)); loc = cc(random.choice(VERB), random.choice(ATTR))
        call = random.choice(helpers) if helpers else (meth, p)
        L += [f"    public int {meth}(int {p}) {{",
              f"        int {loc} = {call[0]}({p}) + {random.choice(fields)};",
              f"        for (int {cc('scan','index')} = 0; {cc('scan','index')} < {p}; {cc('scan','index')}++) {{",
              f"            {loc} += {random.choice(fields)} % {random.randint(2, 7)};",
              f"        }}",
              f"        if ({loc} > {random.randint(10, 99)}) {{ {loc} -= {random.randint(1, 9)}; }}",
              f"        return {loc};",
              f"    }}"]
    L.append("}")
    return "\n".join(L)

def gen_python(i):
    noun = random.choice(NOUN); cls = cap(noun) + random.choice(["Service","Manager","Engine","Coordinator"])
    fields = [snake(noun, random.choice(ATTR)) for _ in range(random.randint(3, 5))]
    L = [f"# domain service for {noun} handling", f"class {cls}:", "    def __init__(self):"]
    for f in fields: L.append(f"        self.{f} = 0")
    helpers = []
    for _ in range(random.randint(2, 3)):
        hv, hn = random.choice(VERB), random.choice(NOUN); hm = snake(hv, hn)
        p = snake(hn, random.choice(ATTR)); loc = snake(random.choice(VERB), random.choice(ATTR))
        helpers.append((hm, p))
        L += [f"    def {hm}(self, {p}):", f"        {loc} = {p} + self.{random.choice(fields)}",
              f"        return {loc} % {random.randint(3, 9)}"]
    for _ in range(random.randint(3, 5)):
        v, n2 = random.choice(VERB), random.choice(NOUN); meth = snake(v, n2)
        p = snake(n2, random.choice(ATTR)); loc = snake(random.choice(VERB), random.choice(ATTR))
        call = random.choice(helpers) if helpers else (meth, p)
        L += [f"    def {meth}(self, {p}):",
              f"        {loc} = self.{call[0]}({p}) + self.{random.choice(fields)}",
              f"        for {snake('scan','index')} in range({p}):",
              f"            {loc} += self.{random.choice(fields)} % {random.randint(2, 7)}",
              f"        if {loc} > {random.randint(10, 99)}:",
              f"            {loc} -= {random.randint(1, 9)}",
              f"        return {loc}"]
    return "\n".join(L)

GEN = gen_java if LANG == "java" else gen_python

# =========================================================================
# Load carrier files: private dir if given, else generate synthetic.
# =========================================================================
def load_private(path, lang, n):
    ext = ".java" if lang == "java" else ".py"
    files = []
    for p in sorted(Path(path).rglob(f"*{ext}")):
        try:
            t = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        if 20 <= (t.count("\n") + 1) <= 400:
            files.append(t)
        if len(files) >= n:
            break
    return files

if PRIVATE_DIR:
    files = load_private(PRIVATE_DIR, LANG, N_FILES)
    CARRIER = "private"
    if not files:
        raise SystemExit("PRIVATE_DIR set but no suitable files found (check path, extension, and 20-400 line size).")
else:
    files = [GEN(i) for i in range(N_FILES)]
    CARRIER = "synthetic_rich"

# =========================================================================
# Inject synthetic sensitive tokens + C2 identifier masking (same rule family as nb02/nb11)
# =========================================================================
JAVA_KW = set("public private class return if for int void new true false null static else while".split())
PY_KW   = set("def class return if else elif for in while and or not None True False self import from as with".split())
COMMON  = set("String List Map int self range len print get set name id type data value".split())
KW = JAVA_KW if LANG == "java" else PY_KW
def is_code_id(w):
    return ((bool(re.search(r"[a-z][A-Z]", w)) or ("_" in w) or any(c.isdigit() for c in w))
            and w not in KW and w not in COMMON and len(w) >= 3)
def luhn_full(p):
    def tot(s):
        d = [int(c) for c in s]; t = sum(d[-1::-2])
        for x in d[-2::-2]:
            y = x * 2; t += y - 9 if y > 9 else y
        return t % 10
    return next(p + str(c) for c in range(10) if tot(p + str(c)) == 0)
def gen_tokens():
    card = "-".join(luhn_full("".join(str(random.randint(0, 9)) for _ in range(15)))[i:i+4] for i in range(0, 16, 4))
    return {"CARD": card,
            "SECRET": "sk-" + "".join(random.choice("abcdefghijklmnopqrstuvwxyz0123456789") for _ in range(40)),
            "EMAIL": "".join(random.choice("abcdefgh") for _ in range(6)) + "@example.com"}
PAT = {"CARD": re.compile(r"\b(?:\d{4}[- ]?){3}\d{4}\b"),
       "SECRET": re.compile(r"\bsk-[A-Za-z0-9]{20,}\b"),
       "EMAIL": re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b")}
def mask_c2(text):
    toks = gen_tokens()
    cmt = "//" if LANG == "java" else "#"
    fixture = f"\n\n{cmt} injected fixture: {toks['CARD']} {toks['SECRET']} {toks['EMAIL']}\n"
    code = text + fixture
    for t, pat in PAT.items():
        code = pat.sub(lambda m, t=t: f"<{t}_x>", code)
    idmap = {}
    KEEP = re.compile(r"(<[A-Z]+_x>)|([A-Za-z_$][A-Za-z0-9_$]*)")
    def rid(m):
        if m.group(1): return m.group(1)
        w = m.group(2)
        if is_code_id(w):
            k = f"<ID_{hashlib.md5(w.encode()).hexdigest()[:6]}>"; idmap[k] = w; return k
        return w
    return KEEP.sub(rid, code), idmap

ADV_PROMPT = ("You are a security analyst. Placeholders <ID_xxxxxx> replaced identifiers in masked code. "
              "Infer each original identifier from context. Return ONLY a JSON object placeholder->guess.\n"
              "Placeholders: {ph}\n\nCode:\n{code}")
def adv(code, idmap, scoring):
    present = [p for p in idmap if p in code][:K_IDENTS]
    if not present: return None
    g = parse_map(chat(ADV_PROMPT.format(ph=present, code=code), 0.0, tag="recon_C2"))
    if scoring == "exact":
        return sum(1 for p in present if norm(g.get(p, "")) == norm(idmap[p])) / len(present)
    return sum(1 for p in present if near(g.get(p, ""), idmap[p])) / len(present)

def verbatim(nf=12, head=8):
    rs = []
    for t in files[:nf]:
        Ls = t.splitlines()
        if len(Ls) < head * 2: continue
        out = chat("Continue this source file exactly as written. Output only the next lines of code.\n\n"
                   + "\n".join(Ls[:head]), 0.0, tag="verbatim").splitlines()
        rs.append(difflib.SequenceMatcher(None, "\n".join(Ls[head:head*2]), "\n".join(out[:head])).ratio())
    return float(np.mean(rs)) if rs else float("nan")

# =========================================================================
# Public Fineract baseline, matched near-match re-scoring (reuses nb02 artifacts + nb12/xvendor cache if present)
# =========================================================================
def public_baseline_near():
    try:
        m = json.load(open(DATA / "masked_corpus.json", encoding="utf-8"))
        idm = json.load(open(DATA / "id_maps.json", encoding="utf-8"))
    except Exception:
        return None
    pub_c2 = {r["file_id"]: r["code"] for r in m["C2"]}
    fids = sorted(int(f) for f in idm)[:N_FILES]
    vals = []
    for fid in fids:
        code = pub_c2[fid]; idmap = idm[str(fid)]
        present = [p for p in idmap if p in code][:K_IDENTS]
        if not present: continue
        g = parse_map(chat(ADV_PROMPT.format(ph=present, code=code), 0.0, tag="recon_C2_public"))
        vals.append(sum(1 for p in present if near(g.get(p, ""), idmap[p])) / len(present))
    return np.array(vals, float)

# =========================================================================
# Run
# =========================================================================
if not USE_LLM:
    print(f"No key for provider '{PROVIDER}' in .env. Set its key + strong model id and re-run.")
    print("No private code is required for synthetic mode.")
else:
    masks = [mask_c2(t) for t in files]
    vb = verbatim()
    ex = [adv(c, im, "exact") for c, im in masks]; ex = [x for x in ex if x is not None]
    nm = [adv(c, im, "near")  for c, im in masks]; nm = [x for x in nm if x is not None]
    nlo, nhi = boot_ci(nm); elo, ehi = boot_ci(ex)
    pub = public_baseline_near()
    pub_m = float(np.nanmean(pub)) if pub is not None and len(pub) else float("nan")
    mem_share = 1 - (np.nanmean(nm) / pub_m) if pub_m and not np.isnan(pub_m) and pub_m > 0 else float("nan")

    row = {"provider": PROVIDER, "model": MODEL, "carrier": CARRIER, "lang": LANG, "n": len(nm),
           "verbatim_overlap": vb,
           "recovery_exact": float(np.mean(ex)) if ex else float("nan"), "exact_lo": elo, "exact_hi": ehi,
           "recovery_near": float(np.mean(nm)) if nm else float("nan"), "near_lo": nlo, "near_hi": nhi,
           "public_baseline_near": pub_m, "memorization_share": mem_share}
    df = pd.DataFrame([row])
    df.to_csv(TAB / "nonmemorized_carrier.csv", index=False)

    print(f"=== Non-memorized carrier reconstruction (strong adversary: {PROVIDER}/{MODEL}) ===")
    print(f"carrier            : {CARRIER} ({LANG}), N={len(nm)} files")
    print(f"verbatim overlap   : {vb:.3f}   (should be low; high = unexpected memorization/template-predictability)")
    print(f"recovery  (exact)  : {row['recovery_exact']:.3f}  [{elo:.3f}, {ehi:.3f}]")
    print(f"recovery  (near)   : {row['recovery_near']:.3f}  [{nlo:.3f}, {nhi:.3f}]")
    print(f"public baseline nm : {pub_m:.3f}   (matched near-match on Fineract C2)")
    print(f"memorization share : {mem_share:.3f}")
    print("\nRead: near-match recovery on this carrier is the tightened upper end of the [0.006, 0.18] bracket.")
    print("A PRIVATE carrier (richer real context) is the strongest version; synthetic_rich is the key control")
    print("when no private code is available. Compare recovery_near here against the paper's 0.18 (meaning-")
    print("preserving) and 0.006 (original simple synthetic).")


=== Non-memorized carrier reconstruction (strong adversary: openai/gpt-5.4) ===
carrier            : synthetic_rich (java), N=40 files
verbatim overlap   : 0.140   (should be low; high = unexpected memorization/template-predictability)
recovery  (exact)  : 0.025  [0.015, 0.037]
recovery  (near)   : 0.029  [0.017, 0.042]
public baseline nm : 0.724   (matched near-match on Fineract C2)
memorization share : 0.960

Read: near-match recovery on this carrier is the tightened upper end of the [0.006, 0.18] bracket.
A PRIVATE carrier (richer real context) is the strongest version; synthetic_rich is the key control
when no private code is available. Compare recovery_near here against the paper's 0.18 (meaning-
preserving) and 0.006 (original simple synthetic).
